In [1]:
import torchvision
import torchvision.transforms as transforms
# from torchvision.models import resnet50, ResNet50_Weight
from data_utils.cub2011 import Cub2011
import os
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pytorch_influence_functions as ptif
import pandas as pd
import json
from json.decoder import JSONDecodeError
import ast
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

In [2]:
datapath = "/tmp2/IF_manipulation_files/final_files_used_by_ruihan"
dataset = "cifar10"

In [3]:
torch.manual_seed(42)
np.random.seed(42)

    
train_std_features, train_labels = torch.load(f"{datapath}/{dataset}_train.pt")
test_std_features, test_labels = torch.load(f"{datapath}/{dataset}_test.pt")
val_std_features, val_labels = torch.load(f"{datapath}/{dataset}_val.pt")

input_size = train_std_features.shape[1]
num_classes = len(torch.unique(train_labels))
batch_size = 64

In [4]:
# Define your model
class LinearClassifier(nn.Module):
    def __init__(self, input_size=2048, num_classes=num_classes):
        super(LinearClassifier, self).__init__()
        self.linear = nn.Linear(input_size, num_classes)

    def forward(self, x):
        return self.linear(x)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = f'{datapath}/linear_classifier_{dataset}.pt'

# Instantiate the model and move it to GPU if available
model = LinearClassifier(input_size, num_classes).to(device)

criterion = nn.CrossEntropyLoss()

model.to(device)
model.load_state_dict(torch.load(model_path))
model.eval()
test_outputs = model(test_std_features)
predicted_labels = torch.argmax(test_outputs, dim=1)
test_accuracy = torch.sum(predicted_labels == test_labels).item() / len(test_labels)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

Test Accuracy: 92.15%


In [6]:
val_outputs = model(val_std_features)
predicted_labels_val = torch.argmax(val_outputs, dim=1)
val_accuracy = torch.sum(predicted_labels_val == val_labels).item() / len(val_labels)
print(f"Val Accuracy: {val_accuracy * 100:.2f}%")

Val Accuracy: 91.99%


In [7]:
json_file = "accuracies.json"
acc_data = {
        dataset: {
            "val_accuracy": val_accuracy,
            "test_accuracy": test_accuracy
        }
    }
if os.path.exists(json_file):
    try:
        with open(json_file, 'r') as file:
            existing_data = json.load(file)
    except JSONDecodeError:
        existing_data = {}
    existing_data.update(acc_data)
    with open(json_file, 'w') as file:
        json.dump(existing_data, file, indent=4)
else:
    with open(json_file, 'w') as file:
        json.dump(acc_data, file, indent=4)